# 데모 백엔드를 Colab GPU에서 띄우기

런타임 → GPU(T4)로 바꾸고 위에서부터 실행한다. 마지막 셀이 `https://….trycloudflare.com` 주소를 출력하면 그 주소가 데모 화면이다 (프론트도 같은 서버가 내보낸다).
런타임이 끊기면 주소도 사라지므로, 다시 실행하면 새 주소가 나온다.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install -U "transformers>=5,<6" fastapi uvicorn

ROOT = '/content/drive/MyDrive/fact-verification'
!rm -rf /content/fv && git clone -q --depth 1 https://github.com/kohjunseo1/Fact-Verfication.git /content/fv
DEMO = '/content/fv/한기선/demo'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.3.0 which is incompatible.
opentelemetry-semantic-conventions 0.63b1 requires opentelemetry-api==1.42.1, but you have opentelemetry-api 1.45.1 which is incompatible.
diffusers 0.41.0 requires huggingface-hub<2.0,>=1.32.0, but you have huggingface-hub 2.3.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.1 which is incompatible.
opentelemetry-sdk 1.42.1 requires opentelemetry-api==1.42.1, but you have opentelemetry-api 1.45.1 which is incompatible.


In [2]:
# 위키 DB와 FTS5 인덱스는 Drive에서 바로 읽으면 느려서 Colab 로컬 디스크로 복사한다.
# 인덱스는 처음 한 번 여기서 만들고(약 1~2분) Drive에 저장해 둔다
import os
WIKI, INDEX = '/content/wiki_wo_links.db', '/content/wiki_index.db'
if not os.path.exists(WIKI):
    !cp "{ROOT}/data/raw/wiki_wo_links.db" {WIKI}
if not os.path.exists(INDEX):
    if os.path.exists(f'{ROOT}/data/index/wiki_index.db'):
        !cp "{ROOT}/data/index/wiki_index.db" {INDEX}
    else:
        !python3 "{DEMO}/build_index.py" {WIKI} {INDEX}
        !mkdir -p "{ROOT}/data/index" && cp {INDEX} "{ROOT}/data/index/wiki_index.db"

50,000 문서 1s
100,000 문서 2s
150,000 문서 4s
200,000 문서 5s
250,000 문서 6s
300,000 문서 8s
350,000 문서 9s
400,000 문서 11s
450,000 문서 12s
500,000 문서 14s
550,000 문서 16s
600,000 문서 18s
650,000 문서 19s
700,000 문서 21s
750,000 문서 22s
800,000 문서 23s
850,000 문서 25s
900,000 문서 27s
950,000 문서 29s
1,000,000 문서 31s
1,050,000 문서 32s
1,100,000 문서 34s
1,150,000 문서 35s
1,200,000 문서 36s
1,250,000 문서 38s
1,300,000 문서 40s
1,350,000 문서 42s
1,400,000 문서 44s
1,450,000 문서 46s
1,500,000 문서 47s
1,550,000 문서 48s
1,600,000 문서 50s
1,650,000 문서 51s
1,700,000 문서 53s
1,750,000 문서 55s
1,800,000 문서 57s
1,850,000 문서 61s
1,900,000 문서 63s
1,950,000 문서 67s
2,000,000 문서 69s
2,050,000 문서 72s
2,100,000 문서 77s
2,150,000 문서 79s
2,200,000 문서 82s
2,250,000 문서 83s
2,300,000 문서 85s
2,350,000 문서 86s
2,400,000 문서 88s
2,450,000 문서 89s
2,500,000 문서 90s
2,550,000 문서 93s
2,600,000 문서 95s
2,650,000 문서 97s
2,700,000 문서 99s
2,750,000 문서 100s
2,800,000 문서 102s
2,850,000 문서 103s
2,900,000 문서 105s
2,950,000 문서 107s
3,000,000 문서 109s
3,050,000 문서 110s
3,

In [16]:
# 기존 서버를 끈다. server.kill()은 바깥 셸만 끄고 uvicorn이 8000 포트를 잡은 채 남을 수 있어서 pkill로 끈다 ([u]는 이 명령을 돌리는 셸 자신이 걸리지 않게 하려는 것)
!pkill -f "[u]vicorn colab_serve"; sleep 2; pgrep -f "[u]vicorn colab_serve" || echo 서버 꺼짐

서버 꺼짐


In [14]:
# app.py(/predict, /health, /info)를 /api 아래에, frontend/index.html을 / 에 붙인다. k8s의 nginx가 하던 일이다
open(f'{DEMO}/backend/colab_serve.py', 'w').write('''
from fastapi import FastAPI
from fastapi.staticfiles import StaticFiles
import app
root = FastAPI()
root.mount('/api', app.app)
root.mount('/', StaticFiles(directory='../frontend', html=True))
''')

import subprocess, time, urllib.request
env = {**os.environ, 'WIKI_DB': WIKI, 'INDEX_DB': INDEX, 'MODEL_DIR': f'{ROOT}/models'}
server = subprocess.Popen('uvicorn colab_serve:root --port 8000 > /content/server.log 2>&1', shell=True,
                          cwd=f'{DEMO}/backend', env=env)
for _ in range(120):  # 모델 4개를 GPU에 올리는 동안 기다린다
    try:
        urllib.request.urlopen('http://localhost:8000/api/health'); print('서버 준비됨'); break
    except Exception:
        time.sleep(2)
else:
    print(open('/content/server.log').read()[-3000:])

서버 준비됨


In [15]:
# 계정 없이 쓰는 cloudflared 임시 터널로 바깥에 연다
# 이 셀을 다시 돌리면 예전 터널이 살아 있어 옛 주소도 계속 열리므로 먼저 끈다
!pkill -f "[c]loudflared tunnel"
import re
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared
tunnel = subprocess.Popen('/content/cloudflared tunnel --url http://localhost:8000 > /content/tunnel.log 2>&1', shell=True)
for _ in range(30):
    time.sleep(2)
    m = re.search(r'https://[\w-]+\.trycloudflare\.com', open('/content/tunnel.log').read())
    if m:
        print('데모 주소:', m.group()); break

데모 주소: https://intake-administered-parks-converter.trycloudflare.com


In [12]:
# 동작 확인 (GPU면 몇 초 안에 끝난다)
import json
req = urllib.request.Request('http://localhost:8000/api/predict', method='POST', headers={'Content-Type': 'application/json'},
                             data=json.dumps({'claim': 'The Assignation was written by a playwright who became Poet Laureate in 1668.'}).encode())
r = json.load(urllib.request.urlopen(req))
print(r['label'], r['prob_supported'], r['timing'], r['latency_ms'], 'ms')

SUPPORTED 0.916 {'bm25': 1.91, 'rerank': 0.3, 'hop2': 0.51, 'sentence': 0.01, 'verify': 0.01} 2735 ms


In [9]:
# 푸시한 코드를 실행 중인 세션에 받아온다 (주소는 그대로)
# 프론트(index.html)만 바뀌었으면 이것만 실행하고 브라우저에서 강력 새로고침(Cmd+Shift+R)
# 백엔드(app.py)가 바뀌었으면 서버 끄기 셀 → 서버 셀을 다시 실행. 터널 셀은 다시 돌리지 않는다(주소가 바뀐다)
!git -C /content/fv pull -q && git -C /content/fv log --oneline -1

b4aa74a (HEAD -> main, origin/main, origin/HEAD) 한기선 데모 디자인 추가
